# Unit 2: BoardShim 基础 —— 连接与数据流

## 学习目标
- 掌握 `BoardShim.get_board_descr()` 获取板卡信息
- 理解数据数组的行列结构（通道 × 采样点）
- 正确提取 EEG 通道、时间戳、Marker 通道
- 学习 `get_board_data()` 与 `get_current_board_data()` 的区别
- 使用 Synthetic Board 进行完整的数据获取练习

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from brainflow.board_shim import (
    BoardShim, BrainFlowInputParams, BoardIds, BrainFlowPresets
)
from brainflow.data_filter import DataFilter, FilterTypes, DetrendOperations

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("模块导入完成")

## 2.1 获取板卡描述信息

`BoardShim.get_board_descr(board_id)` 是最核心的元数据查询方法，
返回一个字典，包含该板卡的所有通道布局和参数信息。

In [ ]:
# 查看 Synthetic Board 的完整描述
board_id = BoardIds.SYNTHETIC_BOARD
descr = BoardShim.get_board_descr(board_id)

print("=" * 60)
print(f"板卡名称: {descr.get('name', 'Unknown')}")
print(f"采样率: {descr.get('sampling_rate', 'N/A')} Hz")
print(f"总通道数 (num_rows): {descr.get('num_rows', 'N/A')}")
print("=" * 60)

# 通道索引信息
print(f"\n📌 EEG 通道索引:     {descr.get('eeg_channels', [])}")
print(f"📌 EEG 通道名称:     {descr.get('eeg_names', [])}")
print(f"📌 时间戳通道索引:   {descr.get('timestamp_channel', [])}")
print(f"📌 Marker 通道索引:  {descr.get('marker_channel', [])}")
print(f"📌 包序号通道索引:   {descr.get('package_num_channel', [])}")

# 可选通道（Synthetic Board 可能有 accel/gyro）
for key in ['accel_channels', 'gyro_channels', 'battery_channel', 'eda_channels']:
    val = descr.get(key)
    if val and (isinstance(val, list) and len(val) > 0 or isinstance(val, int)):
        print(f"📌 {key}: {val}")

### 对比不同板卡的描述信息

BrainFlow 的统一抽象意味着只需要更改 `board_id`，就能看到不同硬件的通道布局。

In [ ]:
# 对比不同板卡的 EEG 通道配置
boards_to_compare = [
    ("Synthetic Board", BoardIds.SYNTHETIC_BOARD),
    ("Cyton (8ch)", BoardIds.CYTON_BOARD),
    ("Cyton+Daisy (16ch)", BoardIds.CYTON_DAISY_BOARD),
    ("Ganglion (4ch)", BoardIds.GANGLION_BOARD),
]

print(f"{'板卡':<25} {'通道数':<8} {'采样率':<8} {'EEG通道名'}")
print("-" * 80)
for name, bid in boards_to_compare:
    d = BoardShim.get_board_descr(bid)
    n_channels = len(d.get('eeg_channels', []))
    sr = d.get('sampling_rate', 'N/A')
    names = d.get('eeg_names', [])[:6]  # 只显示前6个
    names_str = ', '.join(names) if names else 'N/A'
    print(f"{name:<25} {n_channels:<8} {str(sr) + ' Hz':<8} {names_str}")

## 2.2 数据数组结构详解

`get_board_data()` 返回的数据是 **(num_rows, num_samples)** 的 2D 数组：

```
Row 0:  包序号 (Package Num)
Row 1-8: EEG 通道 1-8  (EEG data in μV)
Row ... : 其他通道（如有）
Row N-2: 时间戳 (Timestamp, UNIX μs)
Row N-1: Marker 通道 (事件标记)
```

现在我们来实际操作：采集数据并正确分离各个通道。

In [ ]:
# 采集一段数据
params = BrainFlowInputParams()
board = BoardShim(BoardIds.SYNTHETIC_BOARD, params)
board.prepare_session()
board.start_stream()
time.sleep(5)
data = board.get_board_data()
board.stop_stream()
board.release_session()

# 获取板卡描述
descr = BoardShim.get_board_descr(BoardIds.SYNTHETIC_BOARD)

# 正确分离各通道
eeg_channels = descr['eeg_channels']
eeg_names = BoardShim.get_eeg_names(BoardIds.SYNTHETIC_BOARD)
ts_channel = descr['timestamp_channel']
marker_channel = descr['marker_channel']

eeg_data = data[eeg_channels, :]     # (8, N) — 仅 EEG 通道
timestamps = data[ts_channel, :]     # (N,)  — 时间戳
markers = data[marker_channel, :]    # (N,)  — 事件标记

print(f"原始数据形状: {data.shape}")
print(f"EEG 数据形状:  {eeg_data.shape}")
print(f"时间戳形状:    {timestamps.shape}")
print(f"\n时间戳范围: {timestamps[0]:.0f} → {timestamps[-1]:.0f}")
print(f"采集时长:     {(timestamps[-1] - timestamps[0]) / 1e6:.2f} 秒")
print(f"\n各通道数据范围 (μV):")
for i, name in enumerate(eeg_names):
    print(f"  {name}: [{eeg_data[i].min():.1f}, {eeg_data[i].max():.1f}]")

## 2.3 数据可视化

使用 matplotlib 将 EEG 信号绘制出来，观察 Synthetic Board 生成的模拟信号。

In [ ]:
# 绘制所有 EEG 通道
fig, axes = plt.subplots(len(eeg_channels), 1, figsize=(14, 10), sharex=True)

for i, ax in enumerate(axes):
    ax.plot(eeg_data[i, :], linewidth=0.5, color=f'C{i}')
    ax.set_ylabel(f'{eeg_names[i]}', fontsize=9)
    ax.set_ylim(eeg_data[i].min() - 10, eeg_data[i].max() + 10)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('采样点')
fig.suptitle('Synthetic Board — 8 通道 EEG 信号', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 使用 pandas 展示数据表格
df = pd.DataFrame(np.transpose(data))
# 重命名列
col_names = []
for i in range(data.shape[0]):
    if i in eeg_channels:
        idx = eeg_channels.index(i)
        col_names.append(eeg_names[idx])
    elif i == ts_channel:
        col_names.append('Timestamp')
    elif i == marker_channel:
        col_names.append('Marker')
    else:
        col_names.append(f'Ch_{i}')
df.columns = col_names

print("数据预览 (前10行):")
display(df.head(10))
print(f"\n数据统计:")
display(df[eeg_names].describe())

## 2.4 get_board_data() vs get_current_board_data()

这两个方法的关键区别：

| 方法 | 清空缓冲区 | 用途 |
|------|------------|------|
| `get_board_data()` | ✅ 是 | 一次性获取所有数据，处理后清空 |
| `get_current_board_data(n)` | ❌ 否 | 获取最新 n 个样本，保留缓冲区，适合实时处理 |

In [ ]:
# 演示 get_current_board_data —— 不消耗缓冲区
board = BoardShim(BoardIds.SYNTHETIC_BOARD, BrainFlowInputParams())
board.prepare_session()
board.start_stream()
time.sleep(2)

# 获取最新 50 个样本（不清空缓冲区）
data_chunk1 = board.get_current_board_data(50)
print(f"第一次获取后缓冲区样本数: {board.get_board_data_count()}")

# 再次获取（缓冲区仍保留）
data_chunk2 = board.get_current_board_data(50)
print(f"第二次获取后缓冲区样本数: {board.get_board_data_count()}")

time.sleep(0.5)
# 使用 get_board_data 清空缓冲区
all_data = board.get_board_data()
print(f"get_board_data() 后缓冲区样本数: {board.get_board_data_count()}")

board.stop_stream()
board.release_session()

print(f"\nchunk1 形状: {data_chunk1.shape}")
print(f"chunk2 形状: {data_chunk2.shape}")
print(f"all_data 形状: {all_data.shape}")

## 2.5 便捷类方法

BrainFlow 提供了多个类方法来快速获取板卡信息，无需先获取完整 descr：

In [ ]:
board_id = BoardIds.SYNTHETIC_BOARD

# 类方法 —— 无需实例化即可调用
print(f"采样率:       {BoardShim.get_sampling_rate(board_id)} Hz")
print(f"EEG 通道:     {BoardShim.get_eeg_channels(board_id)}")
print(f"EEG 通道名:   {BoardShim.get_eeg_names(board_id)}")
print(f"时间戳通道:   {BoardShim.get_timestamp_channel(board_id)}")
print(f"Marker 通道:  {BoardShim.get_marker_channel(board_id)}")
print(f"包序号通道:   {BoardShim.get_package_num_channel(board_id)}")
print(f"板卡预设:     {BoardShim.get_board_presets(board_id)}")

# 所有 EXG 通道（EEG + EMG + ECG）
try:
    print(f"EXG 通道:     {BoardShim.get_exg_channels(board_id)}")
except Exception as e:
    print(f"EXG 通道:     不支持 ({e})")

## 2.6 Session 生命周期异常处理

在实际开发中，应当确保即使发生异常也能正确释放资源。

In [ ]:
def safe_session_demo():
    """演示带异常处理的会话管理"""
    board = None
    try:
        board = BoardShim(BoardIds.SYNTHETIC_BOARD, BrainFlowInputParams())
        board.prepare_session()
        board.start_stream()
        time.sleep(3)
        data = board.get_board_data()
        print(f"✅ 成功采集 {data.shape[1]} 个样本")
        return data
    except Exception as e:
        print(f"❌ 错误: {e}")
    finally:
        if board is not None:
            try:
                if board.is_prepared():
                    board.stop_stream()
                    board.release_session()
                    print("🔒 资源已安全释放")
            except Exception:
                pass

_ = safe_session_demo()

## 2.7 小练习

用 Synthetic Board 采集 10 秒数据，并完成以下任务：
1. 打印采集到的总样本数
2. 计算理论样本数（采样率 × 时间），与实际的比率
3. 绘制 Fz 和 Cz 两个通道的对比图
4. 计算 Fz 通道数据的均值和标准差

In [ ]:
# === 你的练习代码 ===

board = BoardShim(BoardIds.SYNTHETIC_BOARD, BrainFlowInputParams())
board.prepare_session()
board.start_stream()
time.sleep(10)
data = board.get_board_data()
board.stop_stream()
board.release_session()

descr = BoardShim.get_board_descr(BoardIds.SYNTHETIC_BOARD)
eeg_channels = descr['eeg_channels']
eeg_names = BoardShim.get_eeg_names(BoardIds.SYNTHETIC_BOARD)
sampling_rate = descr['sampling_rate']

# 1. 总样本数
n_samples = data.shape[1]
print(f"实际采集样本数: {n_samples}")

# 2. 理论样本数
theoretical = sampling_rate * 10
print(f"理论样本数:     {theoretical}")
print(f"比率:           {n_samples / theoretical:.2%}")

# 3. Fz vs Cz 对比图
fz_idx = eeg_names.index('Fz')
cz_idx = eeg_names.index('Cz')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(data[eeg_channels[fz_idx], :500], label='Fz', alpha=0.8)
ax.plot(data[eeg_channels[cz_idx], :500], label='Cz', alpha=0.8)
ax.legend()
ax.set_title('Fz vs Cz (前500个样本)')
ax.grid(True, alpha=0.3)
plt.show()

# 4. Fz 均值与标准差
fz_data = data[eeg_channels[fz_idx], :]
print(f"\nFz 通道统计: 均值={fz_data.mean():.2f} μV, 标准差={fz_data.std():.2f} μV")

## 小结

| 知识点 | 要点 |
|--------|------|
| `get_board_descr()` | 获取板卡完整元数据（通道布局、采样率等） |
| 数据数组 | (num_rows, num_samples)，用通道索引提取 |
| `get_board_data()` | 获取全部数据，清空缓冲区 |
| `get_current_board_data(n)` | 获取最新 n 样本，不清空缓冲区 |
| 类方法 | `get_eeg_channels()`, `get_sampling_rate()` 等 |
| 异常安全 | 使用 try/finally 确保释放资源 |

→ [Unit 3: 数据采集深入](unit3_data_acquisition.ipynb) — 学习不同板卡类型、CSV回放、流式传输